# **Building Multi-Tenancy RAG System with LlamaIndex**

![My Image](https://cdn.sanity.io/images/7m9jw85w/production/9919bbf6b993de31cb11bf9a66036b465ea8b380-3840x2144.png?w=3840)


In this notebook you will look into building Multi-Tenancy RAG System using LlamaIndex.

1. Setup
2. Download Data
3. Load Data
4. Create Index
5. Create Ingestion Pipeline
6. Update Metadata and Insert documents
7. Define Query Engines for each user
8. Querying

## Setup

 You should ensure you have `llama-index` and `pypdf` is installed.

In [1]:
!pip install -q llama-index pypdf llama-index-llms-google-genai llama-index-embeddings-google-genai llama-index-readers-file

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 553.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 627.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.6/164.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cpu requires setuptools<82, but you have setuptools 83.0.0 which is incompatible.


### Set Gemini Api Key

In [2]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

In [3]:
from llama_index.core import VectorStoreIndex
from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter
from llama_index.core import SimpleDirectoryReader
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter

from IPython.display import HTML

In [ ]:
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core import Settings

# Configure LlamaIndex to use Gemini
llm = GoogleGenAI(
    model="gemini-3.5-flash",
    api_key=api_key
)

embed_model = GoogleGenAIEmbedding(
    model_name="gemini-embedding-2",
    api_key=api_key
)

Settings.llm = llm
Settings.embed_model = embed_model

## Download Data

We will use `An LLM Compiler for Parallel Function Calling` and `Dense X Retrieval: What Retrieval Granularity Should We Use?` papers for the demonstartions.

In [4]:
!wget --user-agent "Mozilla" "https://arxiv.org/pdf/2312.04511.pdf" -O "llm_compiler.pdf"
!wget --user-agent "Mozilla" "https://arxiv.org/pdf/2312.06648.pdf" -O "dense_x_retrieval.pdf"

--2026-08-08 13:48:54--  https://arxiv.org/pdf/2312.04511.pdf
Resolving arxiv.org (arxiv.org)... 151.101.67.42, 151.101.131.42, 151.101.195.42, ...
Connecting to arxiv.org (arxiv.org)|151.101.67.42|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: /pdf/2312.04511 [following]
--2026-08-08 13:48:54--  https://arxiv.org/pdf/2312.04511
Reusing existing connection to arxiv.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 1020527 (997K) [application/pdf]
Saving to: ‘llm_compiler.pdf’

llm_compiler.pdf    100%[===================>] 996.61K  4.37MB/s    in 0.2s    

2026-08-08 13:48:54 (4.37 MB/s) - ‘llm_compiler.pdf’ saved [1020527/1020527]

--2026-08-08 13:48:54--  https://arxiv.org/pdf/2312.06648.pdf
Resolving arxiv.org (arxiv.org)... 151.101.67.42, 151.101.131.42, 151.101.195.42, ...
Connecting to arxiv.org (arxiv.org)|151.101.67.42|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: /pdf/2312.06

## Load Data


In [5]:
reader = SimpleDirectoryReader(input_files=["/content/dense_x_retrieval.pdf"])
documents_jerry = reader.load_data()

reader = SimpleDirectoryReader(input_files=["/content/llm_compiler.pdf"])
documents_ravi = reader.load_data()

## Create an Empty Index

In [6]:
# Initialize an empty index
index = VectorStoreIndex.from_documents(documents=[])

## Create Ingestion Pipeline

In [7]:
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=512, chunk_overlap=20),
    ]
)

## Update Metadata and Insert Documents

In [8]:
for document in documents_jerry:
    document.metadata["user"] = "Jerry"

nodes = pipeline.run(documents=documents_jerry)
# Insert nodes into the index
index.insert_nodes(nodes)

In [10]:
import time
for document in documents_ravi:
    document.metadata["user"] = "Ravi"

# Adding a delay to avoid 429 Resource Exhausted errors on the Gemini Free Tier
print("Waiting to avoid rate limits...")
time.sleep(100)

nodes = pipeline.run(documents=documents_ravi)
index.insert_nodes(nodes)
print("Ravi's documents indexed successfully.")

Waiting to avoid rate limits...
Ravi's documents indexed successfully.


## Define Query Engines

Define query engines for both the users with necessary filters.

In [11]:
# For Jerry
jerry_query_engine = index.as_query_engine(
    filters=MetadataFilters(
        filters=[
            ExactMatchFilter(
                key="user",
                value="Jerry",
            )
        ]
    ),
    similarity_top_k=3,
)

# For Ravi
ravi_query_engine = index.as_query_engine(
    filters=MetadataFilters(
        filters=[
            ExactMatchFilter(
                key="user",
                value="Ravi",
            )
        ]
    ),
    similarity_top_k=3,
)

## Querying

In [12]:
import nest_asyncio
nest_asyncio.apply()


# Jerry has Dense X Retrieval paper and should be able to answer following question.
response = jerry_query_engine.query(
    "what are propositions mentioned in the paper?"
)
# Print response
display(HTML(f'<p style="font-size:20px">{response.response}</p>'))


In [13]:
# Ravi has LLMCompiler paper
response = ravi_query_engine.query("what are steps involved in LLMCompiler?")

# Print response
display(HTML(f'<p style="font-size:20px">{response.response}</p>'))

In [14]:
# This should not be answered as Jerry does not have information about LLMCompiler
response = jerry_query_engine.query("what are steps involved in LLMCompiler?")

# Print response
display(HTML(f'<p style="font-size:20px">{response.response}</p>'))